In [9]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/qs/lrnxbwzx0fx34kp8dn5jgcnh0000gn/T/ipykernel_12454/1573927475.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [11]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Machine_Learning_Roadmap.pdf
  ✓ Loaded 2 pages

Processing: Mani_vardhan_cv.pdf
  ✓ Loaded 1 pages

Processing: phase2_learning_guide.pdf
  ✓ Loaded 17 pages

Total documents loaded: 20


In [12]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-02T07:19:35+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-02T07:19:35+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/Machine_Learning_Roadmap.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Machine_Learning_Roadmap.pdf', 'file_type': 'pdf'}, page_content='Machine Learning End-to-End Roadmap\nModule 1: Foundations\nLearn Python, NumPy, Pandas, statistics, probability, and linear algebra.\nKey concepts: vectors, matrices, derivatives, probability distributions, Bayes theorem.\nFocus on intuition and examples.\nModule 2: Supervised Learning\nAlgorithms: Linear Regression, Logistic Regression, KNN, Decision Trees, Random Forest, SVM,\nNaive Bayes.\nEach includes intuition, math, use cases, advantages, disadvantages, and Python implementation.\nModule 3: Unsuperv

# embedding and vectorDB

In [13]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [14]:
chunks=split_documents(all_pdf_documents)
chunks

Split 20 documents into 38 chunks

Example chunk:
Content: Machine Learning End-to-End Roadmap
Module 1: Foundations
Learn Python, NumPy, Pandas, statistics, probability, and linear algebra.
Key concepts: vectors, matrices, derivatives, probability distributi...
Metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-02T07:19:35+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-02T07:19:35+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/Machine_Learning_Roadmap.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Machine_Learning_Roadmap.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-02T07:19:35+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-02T07:19:35+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/Machine_Learning_Roadmap.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Machine_Learning_Roadmap.pdf', 'file_type': 'pdf'}, page_content='Machine Learning End-to-End Roadmap\nModule 1: Foundations\nLearn Python, NumPy, Pandas, statistics, probability, and linear algebra.\nKey concepts: vectors, matrices, derivatives, probability distributions, Bayes theorem.\nFocus on intuition and examples.\nModule 2: Supervised Learning\nAlgorithms: Linear Regression, Logistic Regression, KNN, Decision Trees, Random Forest, SVM,\nNaive Bayes.\nEach includes intuition, math, use cases, advantages, disadvantages, and Python implementation.\nModule 3: Unsuperv

In [4]:
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List ,Dict, Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8985.30it/s]


Model loaded successfully. Embedding dimension: 384


# VectorStore

In [6]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [15]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-02T07:19:35+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-02T07:19:35+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/Machine_Learning_Roadmap.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Machine_Learning_Roadmap.pdf', 'file_type': 'pdf'}, page_content='Machine Learning End-to-End Roadmap\nModule 1: Foundations\nLearn Python, NumPy, Pandas, statistics, probability, and linear algebra.\nKey concepts: vectors, matrices, derivatives, probability distributions, Bayes theorem.\nFocus on intuition and examples.\nModule 2: Supervised Learning\nAlgorithms: Linear Regression, Logistic Regression, KNN, Decision Trees, Random Forest, SVM,\nNaive Bayes.\nEach includes intuition, math, use cases, advantages, disadvantages, and Python implementation.\nModule 3: Unsuperv

In [17]:

## convert text to mebeeding
texts =[doc.page_content for doc in chunks]
texts

###genrate 
embeddings= embedding_manager.generate_embeddings(texts)

## vector stire
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 38 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.10s/it]

Generated embeddings with shape: (38, 384)
Adding 38 documents to vector store...
Successfully added 38 documents to vector store
Total documents in collection: 38


# Retriever Pipeline From VectorStore

In [18]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

# RAG Pipeline- VectorDB To LLM Output Generation

In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))



your_groq_api_key_here


In [32]:
from langchain_groq import ChatGroq
# intilze llm
llm =ChatGroq(model="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

# simple rag function:retrive + generate response
def rag_simple(query, retriver, llm , top_k=3):
    reuslts = retriver.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in  reuslts]) if reuslts else ''
    if not context:
        return "No relevent context found to answer the question"

    #    generate the answer use the llm
    prompts=f""" 
            context:
            {context}

            Question:{query}
      
            Answer:"""

    response = llm.invoke([prompts.format(context=context,query=query)])   
    return response.content



In [52]:
answer=rag_simple("what is xgboost",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is xgboost'
Top K: 1, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


XGBoost (Extreme Gradient Boosting) is a popular, open-source, gradient boosting library used for building and training machine learning models, particularly decision trees and random forests. It is designed to be highly efficient, flexible, and scalable, making it a widely-used tool in data science and machine learning.

In the context of the provided code, XGBoost is used for regression tasks, specifically with the `XGBRegressor` class, which is a type of gradient boosting model. The code demonstrates how to use XGBoost to train a model on the California housing dataset, tune its hyperparameters (such as the number of estimators, learning rate, and maximum depth), and evaluate its performance using metrics like mean squared error (MSE) and R-squared (R2).

XGBoost is known for its strengths, including:

1. **Handling large datasets**: XGBoost is optimized for performance and can handle large datasets with millions of examples.
2. **Fast training**: XGBoost uses a variety of technique